# Qwen2.5-7B

In [1]:
import json
import time
import sys
from urllib.request import Request, urlopen

print("Python del kernel:", sys.version.split()[0])

# Caso de prueba
mensajes = [
    {
        "role": "system",
        "content": (
            "Justifica semánticamente una predicción de coautoría ya recibida. "
            "Usa solo la evidencia proporcionada. No cambies la predicción ni "
            "atribuyas al predictor causas que no estén documentadas. "
            "Responde en español con cuatro apartados: Evidencia, "
            "Interpretación, Limitaciones y Conclusión. "
            "Utiliza los identificadores de evidencia en tus afirmaciones."
        )
    },
    {
        "role": "user",
        "content": (
            "Caso S-001, completamente sintético.\n"
            "Autores: Autor A y Autor B.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Puntuación y modelo predictor: no disponibles.\n\n"
            "[E1] Hypothetical article by Author A: "
            "The study investigates dense retrieval of academic abstracts "
            "and evaluates 120 documents.\n"
            "[E2] Hypothetical article by Author B: "
            "The study investigates BM25 retrieval of scientific abstracts "
            "and evaluates 80 documents.\n\n"
            "Justifica la predicción usando únicamente esta evidencia."
        )
    }
]

# Configuración explícita
solicitud = {
    "model": "qwen2.5:7b",
    "messages": mensajes,
    "stream": False,
    "keep_alive": "10m",
    "options": {
        "num_ctx": 4096,
        "num_predict": 350,
        "temperature": 0
    }
}

# Solicitud al servidor local
peticion = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(solicitud, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio = time.perf_counter()
with urlopen(peticion, timeout=180) as conexion:
    respuesta = json.load(conexion)
tiempo_cliente = time.perf_counter() - inicio

# Mostrar la respuesta y las métricas originales
print("\nRESPUESTA DEL MODELO:\n")
print(respuesta["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_cliente, 3), "s")
for campo in ("total_duration", "load_duration",
              "prompt_eval_duration", "eval_duration"):
    if campo in respuesta:
        print(campo, round(respuesta[campo] / 1e9, 3), "s")

for campo in ("prompt_eval_count", "eval_count",
              "done_reason"):
    print(campo, respuesta.get(campo))


Python del kernel: 3.11.16

RESPUESTA DEL MODELO:

Evidencia:
[E1] Autor A ha publicado un artículo que investiga la recuperación densa de resúmenes académicos y evalúa 120 documentos.
[E2] Autor B ha publicado un artículo que investiga la recuperación de BM25 de resúmenes científicos y evalúa 80 documentos.

Interpretación:
La predicción favorable sugiere que Autor A y Autor B han colaborado en un proyecto que probablemente involucre la evaluación de métodos de recuperación de información, ya que ambos han trabajado con resúmenes académicos o científicos. Autor A ha utilizado técnicas de recuperación densa, mientras que Autor B ha utilizado el algoritmo BM25. La evaluación conjunta de diferentes métodos de recuperación podría ser un indicio de una colaboración en un proyecto de investigación que busca comparar y contrastar estas técnicas.

Limitaciones:
No se dispone de información sobre la puntuación o el modelo predictor utilizado para hacer la predicción. Además, no se conoce el co

In [2]:
import copy

# Copiar la solicitud para conservar intacta la versión original
solicitud_v2 = copy.deepcopy(solicitud)

# Añadir restricciones generales al mensaje de sistema
solicitud_v2["messages"][0]["content"] += (
    "\nReglas adicionales: "
    "Una predicción favorable no demuestra una colaboración previa. "
    "La similitud temática no demuestra que los autores hayan trabajado "
    "juntos, compartan un proyecto o colaboren en el futuro. "
    "No inventes publicaciones conjuntas ni causas del predictor. "
    "Distingue hechos documentados de posibilidades hipotéticas. "
    "Cuando la evidencia solo respalde parcialmente una predicción, "
    "indícalo explícitamente sin modificarla."
)

peticion_v2 = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(solicitud_v2, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio_v2 = time.perf_counter()

with urlopen(peticion_v2, timeout=180) as conexion:
    respuesta_v2 = json.load(conexion)

tiempo_v2 = time.perf_counter() - inicio_v2

print("RESPUESTA V2:\n")
print(respuesta_v2["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_v2, 3), "s")
print("Tokens entrada:", respuesta_v2.get("prompt_eval_count"))
print("Tokens salida:", respuesta_v2.get("eval_count"))
print("Terminación:", respuesta_v2.get("done_reason"))


RESPUESTA V2:

Evidencia:
[E1] Autor A ha publicado un artículo que investiga la recuperación densa de resúmenes académicos y evalúa 120 documentos.
[E2] Autor B ha publicado un artículo que investiga la recuperación BM25 de resúmenes científicos y evalúa 80 documentos.

Interpretación:
La predicción favorable se basa en la similitud temática entre los artículos de Autor A y Autor B, ya que ambos se centran en la recuperación de resúmenes académicos y científicos. Autor A utiliza la recuperación densa, mientras que Autor B utiliza el algoritmo BM25. Aunque no hay evidencia directa de que los autores hayan trabajado juntos, la similitud en el tema y el método de investigación sugiere que podrían tener intereses comunes en el campo de la recuperación de información.

Limitaciones:
1. La predicción no se basa en hechos concretos sobre una colaboración previa o futura entre los autores.
2. La similitud en el tema no garantiza que los autores hayan trabajado juntos o que colaboren en el fut

In [3]:
# Repetir V2 modificando únicamente el límite de salida
solicitud_v2_600 = copy.deepcopy(solicitud_v2)
solicitud_v2_600["options"]["num_predict"] = 600

peticion = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(solicitud_v2_600, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio = time.perf_counter()

with urlopen(peticion, timeout=180) as conexion:
    respuesta_v2_600 = json.load(conexion)

tiempo = time.perf_counter() - inicio

print("RESPUESTA V2 - LÍMITE 600:\n")
print(respuesta_v2_600["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo, 3), "s")
print("Tokens entrada:", respuesta_v2_600.get("prompt_eval_count"))
print("Tokens salida:", respuesta_v2_600.get("eval_count"))
print("Terminación:", respuesta_v2_600.get("done_reason"))
print("Carga:", round(respuesta_v2_600.get("load_duration", 0) / 1e9, 3), "s")


RESPUESTA V2 - LÍMITE 600:

Evidencia:
[E1] Autor A ha publicado un artículo que investiga la recuperación densa de resúmenes académicos y evalúa 120 documentos.
[E2] Autor B ha publicado un artículo que investiga la recuperación BM25 de resúmenes científicos y evalúa 80 documentos.

Interpretación:
La predicción favorable se basa en la similitud temática entre los artículos de Autor A y Autor B, ya que ambos se centran en la recuperación de resúmenes académicos y científicos. Autor A utiliza la recuperación densa, mientras que Autor B utiliza el algoritmo BM25. Aunque no hay evidencia directa de que los autores hayan trabajado juntos, la similitud en el tema y el método de investigación sugiere que podrían tener intereses comunes en el campo de la recuperación de información.

Limitaciones:
1. La predicción no se basa en hechos concretos sobre una colaboración previa o futura entre los autores.
2. La similitud en el tema no garantiza que los autores hayan trabajado juntos o que colabo

In [4]:

# V3: separar explícitamente predicción y justificación
solicitud_v3 = copy.deepcopy(solicitud_v2_600)

solicitud_v3["messages"][0]["content"] += (
    "\nRegla fundamental: distingue siempre entre la predicción "
    "recibida y el respaldo académico que proporciona la evidencia. "
    "No atribuyas la predicción a similitudes temáticas, afiliaciones, "
    "atributos del grafo ni otros factores salvo que se documente "
    "explícitamente su influencia en el predictor. "
    "La evidencia puede respaldar la pertinencia académica de una "
    "colaboración sugerida sin explicar cómo el predictor obtuvo "
    "su resultado. Conserva el carácter sintético del caso."
)

peticion_v3 = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(solicitud_v3, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio_v3 = time.perf_counter()

with urlopen(peticion_v3, timeout=180) as conexion:
    respuesta_v3 = json.load(conexion)

tiempo_v3 = time.perf_counter() - inicio_v3

print("RESPUESTA V3:\n")
print(respuesta_v3["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_v3, 3), "s")
print("Tokens entrada:", respuesta_v3.get("prompt_eval_count"))
print("Tokens salida:", respuesta_v3.get("eval_count"))
print("Terminación:", respuesta_v3.get("done_reason"))


RESPUESTA V3:

Evidencia:
[E1] Autor A ha publicado un artículo que investiga la recuperación densa de resúmenes académicos y evalúa 120 documentos.
[E2] Autor B ha publicado un artículo que investiga la recuperación BM25 de resúmenes científicos y evalúa 80 documentos.

Interpretación:
La predicción favorable se basa en la similitud temática entre los trabajos de los autores, ya que ambos se centran en la recuperación de resúmenes académicos y científicos. Autor A ha trabajado con un enfoque de recuperación densa, mientras que Autor B ha utilizado el algoritmo BM25. Ambos autores han evaluado un conjunto significativo de documentos, lo que sugiere una experiencia en la evaluación de sistemas de recuperación de información.

Limitaciones:
La evidencia proporcionada no demuestra que los autores hayan trabajado juntos en el pasado ni que tengan planes de colaborar en el futuro. La similitud temática en el tema de los trabajos no es suficiente para concluir que los autores han colaborado 

In [5]:
import json
from datetime import datetime, timezone
from uuid import uuid4

# Variables de las cuatro ejecuciones
pruebas = [
    ("V1", "solicitud", "respuesta", "tiempo_cliente"),
    ("V2", "solicitud_v2", "respuesta_v2", "tiempo_v2"),
    ("V2_600", "solicitud_v2_600", "respuesta_v2_600", "tiempo"),
    ("V3", "solicitud_v3", "respuesta_v3", "tiempo_v3")
]

registros = []

for version, entrada, salida, duracion in pruebas:
    if entrada not in globals() or salida not in globals():
        print("Faltan variables de:", version)
        continue

    registros.append({
        "case_id": "S-001",
        "tipo": "sintetico",
        "version": version,
        "solicitud": globals()[entrada],
        "respuesta_original": globals()[salida],
        "tiempo_cliente_s": globals().get(duracion)
    })

# Guardar solamente si están disponibles las cuatro ejecuciones
if len(registros) == 4:
    fecha = datetime.now(timezone.utc)

    nombre = (
        f"piloto_S001_{fecha:%Y%m%dT%H%M%SZ}_"
        f"{uuid4().hex[:6]}.json"
    )

    with open(nombre, "x", encoding="utf-8") as archivo:
        json.dump({
            "fecha_exportacion_utc": fecha.isoformat(),
            "caso": "S-001",
            "registros": registros
        }, archivo, ensure_ascii=False, indent=2)

    print("Archivo creado:", nombre)
    print("Ejecuciones guardadas:", len(registros))
else:
    print("No se guardó el archivo: faltan ejecuciones.")


Archivo creado: piloto_S001_20261031T083303Z_4870ed.json
Ejecuciones guardadas: 4


## Caso S-002

In [6]:
import copy
import json
import time
from datetime import datetime, timezone
from uuid import uuid4
from urllib.request import Request, urlopen

# Reutilizar el mismo prompt y configuración de V3
solicitud_s002 = copy.deepcopy(solicitud_v3)

# Cambiar únicamente los datos del caso
solicitud_s002["messages"][1]["content"] = (
    "Caso S-002. Completamente sintético.\n"
    "Autores: Autor C y Autor D.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Puntuación y modelo predictor: no disponibles.\n"
    "Versión del grafo: no disponible.\n\n"
    "[E1] Afiliación de Autor C: Unidad Académica X (ficticia).\n"
    "[E2] Afiliación de Autor D: Unidad Académica X (ficticia).\n\n"
    "No se proporcionan artículos, resúmenes, palabras clave "
    "ni datos del grafo.\n"
    "Justifica la predicción con la evidencia disponible."
)

# Preparar registro
registro_s002 = {
    "execution_id": str(uuid4()),
    "case_id": "S-002",
    "grupo": "desarrollo",
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud_s002
}

inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(
            solicitud_s002, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urlopen(peticion, timeout=180) as conexion:
        respuesta_s002 = json.load(conexion)

    registro_s002["estado"] = "ok"
    registro_s002["respuesta_original"] = respuesta_s002

except Exception as error:
    registro_s002["estado"] = "error"
    registro_s002["error"] = str(error)

registro_s002["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

# Guardar tanto ejecuciones correctas como fallidas
nombre = (
    f"piloto_S002_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_"
    f"{uuid4().hex[:6]}.json"
)

with open(nombre, "x", encoding="utf-8") as archivo:
    json.dump(registro_s002, archivo, ensure_ascii=False, indent=2)

print("Archivo:", nombre)
print("Estado:", registro_s002["estado"])

if registro_s002["estado"] == "ok":
    print("\nRESPUESTA:\n")
    print(respuesta_s002["message"]["content"])

    print("\nMÉTRICAS:")
    print("Tiempo cliente:", registro_s002["tiempo_cliente_s"], "s")
    print("Tokens entrada:", respuesta_s002.get("prompt_eval_count"))
    print("Tokens salida:", respuesta_s002.get("eval_count"))
    print("Terminación:", respuesta_s002.get("done_reason"))
else:
    print("Error:", registro_s002["error"])


Archivo: piloto_S002_20261031T083308Z_744ecc.json
Estado: ok

RESPUESTA:

Evidencia:
[E1] Autor C está afiliado a la Unidad Académica X.
[E2] Autor D también está afiliado a la Unidad Académica X.

Interpretación:
La evidencia sugiere que ambos autores comparten una afiliación institucional en común, la Unidad Académica X. Esta similitud en la afiliación podría indicar que los autores tienen un contexto de trabajo similar y podrían tener oportunidades para interactuar y colaborar. Sin embargo, la predicción favorable no se basa en la colaboración previa o en proyectos conjuntos, sino en la posibilidad de que la afiliación compartida aumente las posibilidades de futura colaboración.

Limitaciones:
1. No se dispone de información sobre la naturaleza de la afiliación (por ejemplo, si son miembros del mismo equipo de investigación, tienen intereses de investigación similares, etc.).
2. La predicción favorable no se respalda con datos sobre publicaciones, proyectos o interacciones pasadas.


In [7]:

# Reutilizar configuración V3 sin modificar el prompt
solicitud_s003 = copy.deepcopy(solicitud_v3)

solicitud_s003["messages"][1]["content"] = (
    "Caso S-003. Completamente sintético.\n"
    "Autores: Autor E y Autor F.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n\n"
    "[E1] Ficha bibliográfica A del artículo P-E01, Autor E, "
    "año 2024. Resumen: 'We compare dense retrieval and "
    "BM25 using 120 scientific abstracts.'\n\n"
    "[E2] Ficha bibliográfica B del mismo artículo P-E01, "
    "Autor E, año 2024. Resumen: 'We compare dense retrieval "
    "and BM25 using 80 scientific abstracts.'\n\n"
    "[E3] Artículo P-F01, Autor F, año 2025. "
    "Resumen: 'We study query expansion methods for "
    "retrieving academic abstracts.'\n\n"
    "Las fichas E1 y E2 tienen la misma procedencia temporal, "
    "pero no se dispone de información para determinar "
    "cuál cifra es correcta.\n\n"
    "Justifica la predicción con la evidencia disponible."
)

registro_s003 = {
    "execution_id": str(uuid4()),
    "case_id": "S-003",
    "grupo": "desarrollo",
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud_s003
}

inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(
            solicitud_s003, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urlopen(peticion, timeout=180) as conexion:
        respuesta_s003 = json.load(conexion)

    registro_s003["estado"] = "ok"
    registro_s003["respuesta_original"] = respuesta_s003

except Exception as error:
    registro_s003["estado"] = "error"
    registro_s003["error"] = str(error)

registro_s003["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

nombre = (
    f"piloto_S003_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_"
    f"{uuid4().hex[:6]}.json"
)

with open(nombre, "x", encoding="utf-8") as archivo:
    json.dump(registro_s003, archivo, ensure_ascii=False, indent=2)

print("Archivo:", nombre)
print("Estado:", registro_s003["estado"])

if registro_s003["estado"] == "ok":
    print("\nRESPUESTA:\n")
    print(respuesta_s003["message"]["content"])
    print("\nMÉTRICAS:")
    print("Tiempo cliente:", registro_s003["tiempo_cliente_s"], "s")
    print("Tokens entrada:", respuesta_s003.get("prompt_eval_count"))
    print("Tokens salida:", respuesta_s003.get("eval_count"))
    print("Terminación:", respuesta_s003.get("done_reason"))
else:
    print("Error:", registro_s003["error"])

Archivo: piloto_S003_20261031T083314Z_1fa7a5.json
Estado: ok

RESPUESTA:

### Evidencia
La evidencia disponible incluye dos resúmenes del mismo artículo (P-E01) publicado en 2024 por Autor E, y un artículo (P-F01) publicado en 2025 por Autor F. Los resúmenes de los artículos de Autor E se centran en la comparación de métodos de recuperación densa y BM25, mientras que el artículo de Autor F se centra en el estudio de métodos de expansión de consultas para la recuperación de resúmenes académicos.

### Interpretación
Aunque los artículos de Autor E y Autor F se centran en temas relacionados con la recuperación de información en bases de datos de resúmenes académicos, no hay evidencia directa que indique que los autores hayan trabajado juntos o que se prevé una colaboración futura. La similitud temática no es suficiente para respaldar una predicción favorable de coautoría.

### Limitaciones
La principal limitación es que no se dispone de información sobre el modelo predictor, su puntuación

In [8]:
# Caso S-004: evidencia poco pertinente
solicitud_s004 = copy.deepcopy(solicitud_v3)

solicitud_s004["messages"][1]["content"] = (
    "Caso S-004. Completamente sintético.\n"
    "Autores: Autor G y Autor H.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n\n"
    "[E1] Artículo P-G01, Autor G, año 2024. "
    "Abstract: 'We investigate static analysis techniques "
    "for detecting race conditions in concurrent programs. "
    "The evaluation uses 64 code samples.'\n\n"
    "[E2] Artículo P-H01, Autor H, año 2025. "
    "Abstract: 'We investigate convolutional neural networks "
    "for underwater fish image classification. "
    "The evaluation covers 35 image classes.'\n\n"
    "Justifica la predicción con la evidencia disponible."
)

registro_s004 = {
    "execution_id": str(uuid4()),
    "case_id": "S-004",
    "grupo": "desarrollo",
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud_s004
}

inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(solicitud_s004, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urlopen(peticion, timeout=180) as conexion:
        respuesta_s004 = json.load(conexion)

    registro_s004["estado"] = "ok"
    registro_s004["respuesta_original"] = respuesta_s004

except Exception as error:
    registro_s004["estado"] = "error"
    registro_s004["error"] = str(error)

registro_s004["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

nombre = (
    f"piloto_S004_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_"
    f"{uuid4().hex[:6]}.json"
)

with open(nombre, "x", encoding="utf-8") as archivo:
    json.dump(registro_s004, archivo, ensure_ascii=False, indent=2)

print("Archivo:", nombre)
print("Estado:", registro_s004["estado"])

if registro_s004["estado"] == "ok":
    print("\nRESPUESTA:\n")
    print(respuesta_s004["message"]["content"])
    print("\nMÉTRICAS:")
    print("Tiempo cliente:", registro_s004["tiempo_cliente_s"], "s")
    print("Tokens entrada:", respuesta_s004.get("prompt_eval_count"))
    print("Tokens salida:", respuesta_s004.get("eval_count"))
    print("Terminación:", respuesta_s004.get("done_reason"))
else:
    print("Error:", registro_s004["error"])

Archivo: piloto_S004_20261031T083320Z_53eda0.json
Estado: ok

RESPUESTA:

Evidencia:
[E1] Artículo P-G01, Autor G, año 2024, se centra en técnicas de análisis estático para detectar condiciones de carrera en programas concurrentes. El artículo evalúa 64 muestras de código.
[E2] Artículo P-H01, Autor H, año 2025, se enfoca en redes neuronales convolucionales para la clasificación de imágenes de peces submarinos. El estudio abarca 35 clases de imágenes.

Interpretación:
La predicción favorable basada en la evidencia proporcionada se basa en la similitud temática entre los artículos de Autor G y Autor H. Ambos trabajos se centran en la investigación de técnicas de análisis y evaluación, aunque en contextos muy diferentes: uno en programación concurrente y el otro en procesamiento de imágenes. Sin embargo, la similitud temática no demuestra necesariamente una colaboración previa o futura entre los autores.

Limitaciones:
1. La evidencia no proporciona información sobre la interacción o con

In [9]:
import json
import csv
from glob import glob

archivos = sorted(glob("piloto_S00[1-4]_*.json"))

print("Archivos encontrados:", len(archivos))
for archivo in archivos:
    print(archivo)

if len(archivos) != 4:
    print("Revisar archivos: se esperaba uno por cada caso.")
else:
    filas = []

    for archivo in archivos:
        with open(archivo, encoding="utf-8") as f:
            datos = json.load(f)

        # S-001 contiene cuatro ejecuciones; los otros, una.
        registros = datos.get("registros", [datos])

        for r in registros:
            solicitud = r["solicitud"]
            respuesta = r.get("respuesta_original", {})
            opciones = solicitud["options"]

            filas.append({
                "case_id": r["case_id"],
                "version_prompt": r.get("version", "V3"),
                "modelo": solicitud["model"],
                "contexto": opciones["num_ctx"],
                "max_tokens": opciones["num_predict"],
                "temperatura": opciones["temperature"],
                "tokens_entrada": respuesta.get("prompt_eval_count"),
                "tokens_salida": respuesta.get("eval_count"),
                "tiempo_cliente_s": r.get("tiempo_cliente_s"),
                "terminacion": respuesta.get("done_reason"),
                "estado": r.get("estado", "ok"),
                "archivo_origen": archivo,
                "respuesta_texto": respuesta.get("message", {}).get("content", "")
            })

    with open("piloto_resultados.csv", "x",
              newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=filas[0].keys())
        escritor.writeheader()
        escritor.writerows(filas)

    print("\nArchivo creado: piloto_resultados.csv")
    print("Ejecuciones registradas:", len(filas))


Archivos encontrados: 4
piloto_S001_20261031T083303Z_4870ed.json
piloto_S002_20261031T083308Z_744ecc.json
piloto_S003_20261031T083314Z_1fa7a5.json
piloto_S004_20261031T083320Z_53eda0.json

Archivo creado: piloto_resultados.csv
Ejecuciones registradas: 7


In [10]:
import csv

# Leer los resultados consolidados
with open("piloto_resultados.csv", encoding="utf-8-sig") as f:
    registros = list(csv.DictReader(f))

# Tomar únicamente la versión V3 de cada caso
casos = [r for r in registros if r["version_prompt"] == "V3"]

columnas = [
    "case_id", "version_prompt",
    "respaldo_referencias_0_2",
    "conservacion_datos_0_2",
    "limites_predictor_0_2",
    "manejo_evidencia_0_2",
    "claridad_formato_0_2",
    "errores_criticos",
    "observaciones",
    "archivo_origen"
]

if len(casos) == 4 and len({r["case_id"] for r in casos}) == 4:
    with open("evaluacion_manual_desarrollo.csv", "x",
              newline="", encoding="utf-8-sig") as f:

        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writeheader()

        for r in casos:
            escritor.writerow({
                "case_id": r["case_id"],
                "version_prompt": r["version_prompt"],
                "archivo_origen": r["archivo_origen"]
            })

    print("Archivo creado: evaluacion_manual_desarrollo.csv")
    print("Casos preparados:", len(casos))
else:
    print("Revisar registros V3 antes de crear la evaluación.")


Archivo creado: evaluacion_manual_desarrollo.csv
Casos preparados: 4


In [11]:
import json
import hashlib

# Casos completamente sintéticos, reservados para comparación
casos = [
    {
        "case_id": "S-005",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor I y Autor J.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Artículo P-I01, Autor I, 2024. "
            "Abstract: 'We study multilingual dense retrieval "
            "for scientific literature using 150 abstracts.'\n"
            "[E2] Artículo P-J01, Autor J, 2025. "
            "Abstract: 'We evaluate multilingual query expansion "
            "for scientific literature search using 75 abstracts.'"
        )
    },
    {
        "case_id": "S-006",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor K y Autor L.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Artículo P-K01, Autor K, 2024. "
            "Abstract: 'We investigate graph query optimization "
            "for academic databases.'\n"
            "No se proporcionaron artículos ni resúmenes de Autor L."
        )
    },
    {
        "case_id": "S-007",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor M y Autor N.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Ficha A del artículo P-M01, Autor M, año 2024. "
            "Resumen: 'The study evaluates 46 graph queries.'\n"
            "[E2] Ficha B del mismo artículo P-M01, Autor M, "
            "año 2025. Resumen: 'The study evaluates 46 graph queries.'\n"
            "[E3] Artículo P-N01, Autor N, 2025. "
            "Resumen: 'We investigate graph-based information retrieval.'\n"
            "No se dispone de información para determinar "
            "cuál año de P-M01 es correcto."
        )
    },
    {
        "case_id": "S-008",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor O y Autor P.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Artículo P-O01, Autor O, 2024. "
            "Abstract: 'We investigate energy-efficient scheduling "
            "in data centers with heterogeneous processors.'\n"
            "[E2] Artículo P-P01, Autor P, 2025. "
            "Abstract: 'We study retinal image segmentation "
            "using convolutional neural networks.'"
        )
    },
    {
        "case_id": "S-009",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor Q y Autor R.\n"
            "Predicción recibida: no favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Artículo P-Q01, Autor Q, 2024. "
            "Abstract: 'We investigate retrieval-augmented generation "
            "for answering scientific questions.'\n"
            "[E2] Artículo P-R01, Autor R, 2025. "
            "Abstract: 'We evaluate evidence retrieval methods "
            "for scientific question answering.'"
        )
    },
    {
        "case_id": "S-010",
        "tipo": "sintetico",
        "grupo": "comparacion",
        "contenido": (
            "Autores: Autor S y Autor T.\n"
            "Predicción recibida: favorable (simulada).\n"
            "Modelo predictor y puntuación: no disponibles.\n"
            "[E1] Artículo P-S09, Autor S, 2024. "
            "Abstract: 'The retrieval system correctly answers "
            "31 out of 47 academic queries.'\n"
            "[E2] Artículo P-T02, Autor T, 2025. "
            "Abstract: 'We evaluate a document retrieval system "
            "using 28 academic queries.'"
        )
    }
]

# Crear el archivo sin sobrescribir archivos anteriores
nombre = "casos_comparacion_v1.json"

with open(nombre, "x", encoding="utf-8") as f:
    json.dump(casos, f, ensure_ascii=False, indent=2)

# Identificador de integridad del archivo
with open(nombre, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("Archivo creado:", nombre)
print("Número de casos:", len(casos))
print("Identificadores:", [c["case_id"] for c in casos])
print("SHA256:", sha256)


Archivo creado: casos_comparacion_v1.json
Número de casos: 6
Identificadores: ['S-005', 'S-006', 'S-007', 'S-008', 'S-009', 'S-010']
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d


In [12]:

import json
import copy
import time
import hashlib
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# 1. Verificar que los casos no hayan cambiado
with open("casos_comparacion_v1.json", "rb") as f:
    contenido = f.read()

sha = hashlib.sha256(contenido).hexdigest()

assert sha == "f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d", \
    "El archivo de casos fue modificado."

casos = json.loads(contenido.decode("utf-8"))

assert "solicitud_v3" in globals(), "Falta recuperar el prompt V3."
assert solicitud_v3["model"] == "qwen2.5:7b"
assert solicitud_v3["options"]["num_ctx"] == 4096
assert solicitud_v3["options"]["num_predict"] == 600
assert solicitud_v3["options"]["temperature"] == 0

# 2. Ejecutar cada caso por separado
for caso in casos:

    solicitud_actual = copy.deepcopy(solicitud_v3)

    solicitud_actual["messages"][1]["content"] = (
        f"Caso {caso['case_id']}. Completamente sintético.\n"
        f"{caso['contenido']}\n\n"
        "Justifica la predicción con la evidencia disponible."
    )

    registro = {
        "execution_id": str(uuid4()),
        "case_id": caso["case_id"],
        "grupo": "comparacion",
        "prompt": "V3",
        "sha256_casos": sha,
        "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
        "solicitud": solicitud_actual,
        "reintentos": 0
    }

    inicio = time.perf_counter()

    try:
        peticion = Request(
            "http://127.0.0.1:11434/api/chat",
            data=json.dumps(
                solicitud_actual, ensure_ascii=False
            ).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST"
        )

        with urlopen(peticion, timeout=180) as conexion:
            respuesta = json.load(conexion)

        registro["respuesta_original"] = respuesta

        if respuesta.get("done_reason") == "length":
            registro["estado"] = "truncada"
        else:
            registro["estado"] = "ok"

    except Exception as error:
        registro["estado"] = "error"
        registro["error"] = str(error)

    registro["tiempo_cliente_s"] = round(
        time.perf_counter() - inicio, 3
    )
    registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

    # 3. Guardar cada ejecución sin sobrescribir archivos
    nombre = (
        f"comparacion_{caso['case_id']}_qwen25_"
        f"{registro['execution_id'][:8]}.json"
    )

    with open(nombre, "x", encoding="utf-8") as f:
        json.dump(registro, f, ensure_ascii=False, indent=2)

    # Mostrar solamente un resumen
    print(
        caso["case_id"],
        "|", registro["estado"],
        "|", registro["tiempo_cliente_s"], "s",
        "|", nombre
    )

print("\nEjecución de los seis casos finalizada.")

S-005 | ok | 6.036 s | comparacion_S-005_qwen25_900c8d4c.json
S-006 | ok | 4.318 s | comparacion_S-006_qwen25_349d5588.json
S-007 | ok | 5.712 s | comparacion_S-007_qwen25_a233f546.json
S-008 | ok | 5.176 s | comparacion_S-008_qwen25_b6920ab6.json
S-009 | ok | 5.825 s | comparacion_S-009_qwen25_b5c56520.json
S-010 | ok | 5.814 s | comparacion_S-010_qwen25_27b4982b.json

Ejecución de los seis casos finalizada.
